In [ ]:
!pip install -q kagglehub

import kagglehub
import os
import shutil


path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")
print("Dataset yuklangan yo'l:", path)

os.makedirs("data", exist_ok=True)
for file_name in os.listdir(path):
    shutil.copy(os.path.join(path, file_name), os.path.join("data", file_name))

print("data/ papkasidagi fayllar:", os.listdir("data"))

Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.
Dataset yuklangan yo'l: /kaggle/input/tmdb-movie-metadata
data/ papkasidagi fayllar: ['tmdb_5000_credits.csv', 'tmdb_5000_movies.csv']


In [ ]:
import pandas as pd

movies = pd.read_csv("data/tmdb_5000_movies.csv")
credits = pd.read_csv("data/tmdb_5000_credits.csv")

print(f"Movies jadvali hajmi: {movies.shape}")   # (4803, 20)
print(f"Credits jadvali hajmi: {credits.shape}") # (4803, 4)

Movies jadvali hajmi: (4803, 20)
Credits jadvali hajmi: (4803, 4)


In [ ]:
# 3-katakcha: Ustunlar ro'yxatini chiqarish
print("--- MOVIES USTUNLARI ---")
print(movies.columns.tolist())

print("\n--- CREDITS USTUNLARI ---")
print(credits.columns.tolist())

--- MOVIES USTUNLARI ---
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

--- CREDITS USTUNLARI ---
['movie_id', 'title', 'cast', 'crew']


In [ ]:
# Jadvallarni birlashtirish
df = movies.merge(credits, left_on="id", right_on="movie_id")

print("Birlashgan jadval hajmi:", df.shape)
print("Film nomi namunasi:", df.loc[0, "title_x"])

Birlashgan jadval hajmi: (4803, 24)
Film nomi namunasi: Avatar


In [ ]:
# Kerakli ustunlarni ajratish
movies = df[[
    'movie_id',
    'title_x',
    'overview',
    'genres',
    'keywords',
    'cast',
    'crew',
    'release_date'
]].copy()

# Ustun nomini to'g'rilaymiz
movies.rename(columns={'title_x': 'title'}, inplace=True)

# Yilni alohida ustun qilib ajratamiz (masalan: 2014)
movies['release_year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year

# Endi to'liq sana ustuni kerak emas, o'chirib tashlaymiz
movies.drop(columns=['release_date'], inplace=True)

movies.head(3)

,movie_id,title,overview,genres,keywords,cast,crew,release_year
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",2009.0
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",2007.0
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",2015.0


In [ ]:
# Bo'sh qiymatlar va dublikatlarni tekshirish
print("Bo'sh qiymatlar soni:")
print(movies.isnull().sum())

# Overview yoki release_year bo'lmagan sanoqli filmlarni o'chirib tashlaymiz
movies.dropna(subset=['overview', 'release_year'], inplace=True)

# Yilni butun songa (int) aylantiramiz
movies['release_year'] = movies['release_year'].astype(int)

# Bir xil movie_id takrorlangan bo'lsa, bittasini qoldiramiz
movies.drop_duplicates(subset=['movie_id'], inplace=True)

print("\nTozalashdan keyingi umumiy filmlar soni:", len(movies))

Bo'sh qiymatlar soni:
movie_id        0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
release_year    1
dtype: int64

Tozalashdan keyingi umumiy filmlar soni: 4799


In [ ]:
# JSON matnlarni ro'yxatga aylantiruvchi funksiyalar
import ast

def extract_names(obj):
    """Barcha lug'atlardan 'name' qiymatini ajratib oladi (genres va keywords uchun)"""
    names = []
    try:
        for item in ast.literal_eval(obj):
            names.append(item['name'])
    except (ValueError, SyntaxError):
        pass
    return names

def extract_top_cast(obj, limit=3):
    """Faqat dastlabki 3 ta bosh rol aktyorini oladi"""
    cast_names = []
    try:
        for item in ast.literal_eval(obj)[:limit]:
            cast_names.append(item['name'])
    except (ValueError, SyntaxError):
        pass
    return cast_names

def extract_director(obj):
    """Crew ichidan faqat Rejissyor (Director) ismini ajratib oladi"""
    directors = []
    try:
        for item in ast.literal_eval(obj):
            if item.get('job') == 'Director':
                directors.append(item['name'])
                break # Odatda bitta asosiy rejissyor kifoya
    except (ValueError, SyntaxError):
        pass
    return directors

In [ ]:
# Ustunlarni qayta ishlash
movies['genres'] = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast'] = movies['cast'].apply(extract_top_cast)
movies['director'] = movies['crew'].apply(extract_director)

# Eski crew ustunini o'chirib tashlaymiz
movies.drop(columns=['crew'], inplace=True)

# overview matnini ham so'zlar ro'yxatiga (list of words) aylantiramiz
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Natijani tekshiramiz
movies[['title', 'genres', 'director', 'cast']].head(3)

,title,genres,director,cast
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]",[James Cameron],"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]",[Gore Verbinski],"[Johnny Depp, Orlando Bloom, Keira Knightley]"
2,Spectre,"[Action, Adventure, Crime]",[Sam Mendes],"[Daniel Craig, Christoph Waltz, Léa Seydoux]"


In [ ]:
# Streamlit interfeysi uchun asl nomlarni alohida saqlab qolamiz
movies['genres_display'] = movies['genres'].apply(lambda x: ", ".join(x))
movies['director_display'] = movies['director'].apply(lambda x: ", ".join(x))
movies['cast_display'] = movies['cast'].apply(lambda x: ", ".join(x))

# Model uchun so'zlar orasidagi bo'shliqlarni olib tashlash funksiyasi
# Masalan: "Christopher Nolan" -> "ChristopherNolan"
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['director'] = movies['director'].apply(collapse)

movies[['title', 'genres', 'director']].head(3)

,title,genres,director
0,Avatar,"[Action, Adventure, Fantasy, ScienceFiction]",[JamesCameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]",[GoreVerbinski]
2,Spectre,"[Action, Adventure, Crime]",[SamMendes]


In [ ]:
# Barcha ro'yxatlarni birlashtirib 'tags' ustunini yaratish
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['director']
)

# Ro'yxatni probel bilan ajratilgan yagona matnga aylantiramiz va kichik harf qilamiz
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

# Filmlarning asosiy jadvalini tartiblab olamiz
cleaned_df = movies[[
    'movie_id',
    'title',
    'release_year',
    'genres_display',
    'director_display',
    'cast_display',
    'tags'
]].copy()

# Bitta namunaning 'tags' mazmunini tekshirib ko'ramiz
print(f"Film: {cleaned_df.loc[0, 'title']}")
print(f"Tags namunasi:\n{cleaned_df.loc[0, 'tags'][:250]}...")

Film: Avatar
Tags namunasi:
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar space...


In [ ]:
# Matnlarni o'zak holatiga keltirish (Stemming)
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem_text(text):
    """Matndagi har bir so'zni o'zak shakliga keltiradi"""
    stemmed_words = [ps.stem(word) for word in text.split()]
    return " ".join(stemmed_words)

# Funksiyani tags ustuniga qo'llaymiz
cleaned_df['tags'] = cleaned_df['tags'].apply(stem_text)

# Sinov uchun birinchi qatorning dastlabki 100 ta belgisini ko'ramiz
print("Stemming qilingan tags namunasi:")
print(cleaned_df.loc[0, 'tags'][:150])

Stemming qilingan tags namunasi:
in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civ


In [ ]:
# Matnni sonli vektorlar matritsasiga o'tkazish
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

# Har bir film uchun 5000 o'lchamli vektor hosil qilamiz
vectors = cv.fit_transform(cleaned_df['tags']).toarray()

print("Vektorlar matritsasi shakli:", vectors.shape)
print("Ajratib olingan eng mashhur ba'zi so'zlar:", cv.get_feature_names_out()[:15])

Vektorlar matritsasi shakli: (4799, 5000)
Ajratib olingan eng mashhur ba'zi so'zlar: ['000' '007' '10' '100' '11' '12' '13' '14' '15' '16' '17' '18' '18th'
 '18thcenturi' '19']


In [ ]:
# Kosinus o'xshashligi matritsasini hisoblash
from sklearn.metrics.pairwise import cosine_similarity

# Barcha filmlar orasidagi o'xshashlik matritsasi (4800 x 4800)
similarity = cosine_similarity(vectors)

print("O'xshashlik matritsasi o'lchami:", similarity.shape)
print("1-filmning o'ziga nisbatan o'xshashligi:", similarity[0][0])  # Har doim 1.0 chiqadi

O'xshashlik matritsasi o'lchami: (4799, 4799)
1-filmning o'ziga nisbatan o'xshashligi: 1.0000000000000002


In [ ]:
# Kamida 3 ta film asosida tavsiya beruvchi funksiya
import numpy as np

def recommend_movies(selected_titles, df, similarity_matrix, top_n=3):
    """
    selected_titles: Foydalanuvchi tanlagan filmlar ro'yxati (kamida 3 ta)
    top_n: Qaytarilishi kerak bo'lgan yangi tavsiyalar soni
    """
    if len(selected_titles) < 3:
        raise ValueError("Kamida 3 ta film tanlashingiz kerak!")

    selected_indices = []
    for title in selected_titles:
        # Film nomi bo'yicha uning indeksini topamiz
        match = df[df['title'].str.lower() == title.lower()]
        if not match.empty:
            selected_indices.append(match.index[0])
        else:
            print(f"Ogohlantirish: '{title}' bazadan topilmadi.")

    if not selected_indices:
        return []

    # 1. Tanlangan barcha filmlarning o'xshashlik ballari o'rtachasini olamiz
    user_profile_scores = np.mean(similarity_matrix[selected_indices], axis=0)

    # 2. Tanlangan filmlar qayta tavsiya qilinmasligi uchun ularning balini -1 qilamiz
    for idx in selected_indices:
        user_profile_scores[idx] = -1.0

    # 3. Eng yuqori ball to'plagan top_n ta indeksni saralab olamiz
    top_indices = np.argsort(user_profile_scores)[::-1][:top_n]

    # 4. Natijani ma'lumotlar bilan shakllantiramiz
    recommendations = []
    for idx in top_indices:
        recommendations.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'release_year': df.iloc[idx]['release_year'],
            'genres': df.iloc[idx]['genres_display'],
            'director': df.iloc[idx]['director_display'],
            'cast': df.iloc[idx]['cast_display'],
            'match_score': round(float(user_profile_scores[idx]) * 100, 1)
        })

    return recommendations

In [ ]:
# Test sinovi
test_movies = ["Inception", "Interstellar", "The Matrix"]

results = recommend_movies(test_movies, cleaned_df, similarity, top_n=3)

print("Tanlangan filmlar:", test_movies)
print("\nTavsiya etilgan 3 ta mos film:\n" + "="*50)

for i, rec in enumerate(results, 1):
    print(f"{i}. {rec['title']} ({rec['release_year']})")
    print(f"   Moslik foizi: {rec['match_score']}%")
    print(f"   Janr: {rec['genres']}")
    print(f"   Rejissyor: {rec['director']}")
    print(f"   Bosh rollarda: {rec['cast']}\n")

Tanlangan filmlar: ['Inception', 'Interstellar', 'The Matrix']

Tavsiya etilgan 3 ta mos film:
1. The Matrix Revolutions (2003)
   Moslik foizi: 24.5%
   Janr: Adventure, Action, Thriller, Science Fiction
   Rejissyor: Lilly Wachowski
   Bosh rollarda: Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss

2. The Matrix Reloaded (2003)
   Moslik foizi: 18.0%
   Janr: Adventure, Action, Thriller, Science Fiction
   Rejissyor: Lilly Wachowski
   Bosh rollarda: Keanu Reeves, Carrie-Anne Moss, Laurence Fishburne

3. Guardians of the Galaxy (2014)
   Moslik foizi: 17.4%
   Janr: Action, Science Fiction, Adventure
   Rejissyor: James Gunn
   Bosh rollarda: Chris Pratt, Zoe Saldana, Dave Bautista



In [ ]:
# 16-katakcha: Poster yuklovchi funksiya
import requests

def fetch_poster(movie_id, api_key="c543c15d9d3264ff9b31f5b7363aa644"):
    """
    movie_id bo'yicha TMDb API orqali to'liq poster URL manzilini qaytaradi.
    Agar rasm topilmasa yoki xatolik bo'lsa, zaxira rasmni beradi.
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&language=en-US"
    fallback_image = "https://via.placeholder.com/500x750?text=No+Poster+Available"

    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            poster_path = data.get('poster_path')
            if poster_path:
                return f"https://image.tmdb.org/t/p/w500{poster_path}"
    except requests.exceptions.RequestException:
        pass

    return fallback_image

# Sinov uchun Inception (movie_id: 27205) posterini tekshirib ko'ramiz
print("Sinov poster havolasi:", fetch_poster(27205, api_key="c543c15d9d3264ff9b31f5b7363aa644"))

Sinov poster havolasi: https://image.tmdb.org/t/p/w500/xlaY2zyzMfkhk0HSC5VUwzoZPU1.jpg


In [ ]:
# Model va ma'lumotlarni ixchamlab saqlash
import pickle
import numpy as np

# 1. Matritsani float16 ga o'tkazib xotirani tejaymiz (~46 MB)
similarity_optimized = similarity.astype(np.float16)

# 2. Filmlar ro'yxatini Streamlit uchun kerakli ustunlar bilan saqlaymiz
movies_dict = cleaned_df[[
    'movie_id',
    'title',
    'release_year',
    'genres_display',
    'director_display',
    'cast_display'
]].to_dict(orient='records')

# 3. Fayllarni pickle orqali yozamiz
with open('movies_list.pkl', 'wb') as f:
    pickle.dump(movies_dict, f)

with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity_optimized, f)

print("Fayllar muvaffaqiyatli saqlandi:")
print("- movies_list.pkl")
print("- similarity.pkl")

Fayllar muvaffaqiyatli saqlandi:
- movies_list.pkl
- similarity.pkl


In [ ]:
# Fayllarni kompyuterga yuklab olish
from google.colab import files

files.download('movies_list.pkl')
files.download('similarity.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pickle
from scipy import sparse

# 1. Vektorlarni zich array emas, siqilgan siyrak matritsa (csr_matrix) ko'rinishida olamiz
# Agar vectors = cv.fit_transform(...) qilingan bo'lsa, toarray() qilmasdan to'g'ridan-to'g'ri olinadi:
vectors_sparse = cv.fit_transform(cleaned_df['tags'])

# 2. Saqlash (Hajmi bor-yo'g'i ~2-3 MB bo'ladi!)
with open('movies_list.pkl', 'wb') as f:
    pickle.dump(movies_dict, f)

with open('vectors.pkl', 'wb') as f:
    pickle.dump(vectors_sparse, f)

print("Tayyor! vectors.pkl hajmi atigi 2-3 MB bo'ldi.")

Tayyor! vectors.pkl hajmi atigi 2-3 MB bo'ldi.
